# Stage 1 Gatekeeper — ML Model Training, Evaluation, and Model Card

This notebook trains and compares two classic ML models for the Stage 1 gatekeeper classifier.

Expected input file:

- `stage1_gatekeeper_data.csv`

Expected columns:

- `text`: message content
- `label`: binary class label

Label meaning:

- `0` = spam / non-lead
- `1` = real-estate lead

Models compared:

1. TF-IDF + Logistic Regression
2. TF-IDF + Linear SVM

The notebook includes train/test split, imputation, `ColumnTransformer`, full `Pipeline`s, evaluation metrics, confusion matrices, best-model selection, model export with `joblib`, dataset hashing, and a production-style model card exported as Markdown and JSON.


## 1. Optional: upload the dataset in Colab

Run this cell if `stage1_gatekeeper_data.csv` is not already visible in the Colab file browser.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except ImportError:
    print("This upload helper only works inside Google Colab.")


## 2. Imports and configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

DATA_PATH = "stage1_gatekeeper_data.csv"
MODEL_OUTPUT_PATH = "stage1_gatekeeper_best_model.joblib"
MODEL_CARD_MD_PATH = "stage1_gatekeeper_model_card.md"
MODEL_CARD_JSON_PATH = "stage1_gatekeeper_model_card.json"

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_SPLITS = 5

LABEL_NAMES = {
    0: "spam_non_lead",
    1: "lead",
}


## 3. Load and validate the dataset

In [ ]:
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Upload it to the Colab workspace first."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {"text", "label"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df[["text", "label"]].copy()

# Keep only valid binary labels.
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows by text and label:", df.duplicated(subset=["text", "label"]).sum())

display(df.head(10))


## 4. Dataset fingerprint and metadata

This creates a reproducible SHA-256 hash of the exact CSV file used for training.  
The hash goes into the model card and model package so you can trace which dataset produced the trained artifact.

In [ ]:
def sha256_file(path):
    """Return the SHA-256 hash of a file's raw bytes."""
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_dataframe_content(dataframe):
    """
    Return a content hash of the loaded dataframe.
    This is separate from the raw file hash and is useful if line endings or CSV formatting change.
    """
    canonical_csv = dataframe.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(canonical_csv.encode("utf-8")).hexdigest()


dataset_metadata = {
    "dataset_path": DATA_PATH,
    "dataset_file_size_bytes": Path(DATA_PATH).stat().st_size,
    "dataset_raw_sha256": sha256_file(DATA_PATH),
    "dataset_content_sha256": sha256_dataframe_content(df),
    "row_count": int(len(df)),
    "column_names": list(df.columns),
    "label_distribution": {
        str(label): int(count)
        for label, count in df["label"].value_counts().sort_index().items()
    },
    "duplicate_text_label_rows": int(df.duplicated(subset=["text", "label"]).sum()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(dataset_metadata, indent=2))

## 4. Train/test split

The split is stratified so both classes keep the same proportion in train and test.

In [ ]:
X = df[["text"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain label distribution:")
print(y_train.value_counts().sort_index())

print("\nTest label distribution:")
print(y_test.value_counts().sort_index())


## 5. Preprocessing with ColumnTransformer and Pipeline

Even though the dataset is clean, the pipeline includes `SimpleImputer` so missing text values are handled safely in production.

In [ ]:
text_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="",
            ),
        ),
        (
            "flatten",
            FunctionTransformer(
                np.ravel,
                validate=False,
            ),
        ),
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                max_features=10000,
                min_df=2,
                max_df=0.95,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            text_pipeline,
            ["text"],
        )
    ],
    remainder="drop",
)


## 6. Define two model pipelines

In [ ]:
models = {
    "logistic_regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "linear_svc": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LinearSVC(
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

models


## 7. Optional cross-validation on the training set

This gives a more stable estimate before evaluating on the held-out test set.

In [ ]:
cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_rows = []

for model_name, model_pipeline in models.items():
    print(f"Running cross-validation for: {model_name}")

    cv_scores = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
        },
        n_jobs=-1,
    )

    cv_rows.append(
        {
            "model": model_name,
            "cv_accuracy_mean": cv_scores["test_accuracy"].mean(),
            "cv_precision_mean": cv_scores["test_precision"].mean(),
            "cv_recall_mean": cv_scores["test_recall"].mean(),
            "cv_f1_mean": cv_scores["test_f1"].mean(),
        }
    )

cv_results_df = pd.DataFrame(cv_rows).sort_values(
    by="cv_f1_mean",
    ascending=False,
)

display(cv_results_df)


## 8. Train and evaluate on the test set

In [ ]:
test_rows = []
trained_models = {}

for model_name, model_pipeline in models.items():
    print("=" * 80)
    print(f"Training model: {model_name}")
    print("=" * 80)

    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    test_rows.append(
        {
            "model": model_name,
            "test_accuracy": accuracy,
            "test_precision": precision,
            "test_recall": recall,
            "test_f1": f1,
        }
    )

    trained_models[model_name] = model_pipeline

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[LABEL_NAMES[0], LABEL_NAMES[1]],
    )
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

test_results_df = pd.DataFrame(test_rows).sort_values(
    by="test_f1",
    ascending=False,
)

display(test_results_df)


## 9. Pick the best model

The best model is selected by test F1 score. For this gatekeeper task, F1 is a good default because it balances precision and recall.

In [ ]:
best_model_name = test_results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print(f"Best model: {best_model_name}")

display(test_results_df)


## 10. Inspect important terms

This is useful for debugging whether the model learned reasonable lead/non-lead signals.

In [ ]:
def show_top_terms(model_pipeline, top_n=20):
    classifier = model_pipeline.named_steps["classifier"]
    preprocessor_step = model_pipeline.named_steps["preprocessor"]
    vectorizer = preprocessor_step.named_transformers_["text"].named_steps["tfidf"]
    feature_names = vectorizer.get_feature_names_out()

    if not hasattr(classifier, "coef_"):
        print("This classifier does not expose linear coefficients.")
        return

    coefficients = classifier.coef_[0]
    top_positive_indices = np.argsort(coefficients)[-top_n:][::-1]
    top_negative_indices = np.argsort(coefficients)[:top_n]

    positive_terms = pd.DataFrame(
        {
            "term": feature_names[top_positive_indices],
            "coefficient": coefficients[top_positive_indices],
            "class_signal": LABEL_NAMES[1],
        }
    )

    negative_terms = pd.DataFrame(
        {
            "term": feature_names[top_negative_indices],
            "coefficient": coefficients[top_negative_indices],
            "class_signal": LABEL_NAMES[0],
        }
    )

    print("Top lead signals:")
    display(positive_terms)

    print("Top spam/non-lead signals:")
    display(negative_terms)


show_top_terms(best_model, top_n=20)


## 11. Test on custom examples

In [ ]:
custom_examples = pd.DataFrame(
    {
        "text": [
            "Is this apartment still available?",
            "Can I schedule a viewing tomorrow?",
            "Price please",
            "Do you have a two bedroom apartment in Hamra?",
            "Congratulations you won a free gift card claim now",
            "Your account has been suspended click this link",
            "I already found an apartment thanks",
            "Make $5000 per week from home with no experience",
            None,
        ]
    }
)

custom_predictions = best_model.predict(custom_examples)

custom_results = custom_examples.copy()
custom_results["prediction"] = custom_predictions
custom_results["prediction_name"] = custom_results["prediction"].map(LABEL_NAMES)

# Logistic Regression supports probabilities. LinearSVC does not by default.
if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    custom_results["lead_probability"] = best_model.predict_proba(custom_examples)[:, 1]

display(custom_results)


## 12. Save the best model

The exported `joblib` package includes the full preprocessing pipeline and classifier, so inference only needs a dataframe with a `text` column.

In [ ]:
model_package = {
    "model": best_model,
    "model_name": best_model_name,
    "label_mapping": LABEL_NAMES,
    "input_columns": ["text"],
    "dataset_metadata": dataset_metadata,
    "test_results": test_results_df.to_dict(orient="records"),
    "cv_results": cv_results_df.to_dict(orient="records"),
    "training_config": {
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "cv_splits": CV_SPLITS,
        "selection_metric": "test_f1",
    },
    "runtime_metadata": {
        "python_version": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    },
}

joblib.dump(model_package, MODEL_OUTPUT_PATH)

print(f"Saved best model package to: {MODEL_OUTPUT_PATH}")


## 13. Generate a production-style model card

The model card records what the model is, what data produced it, how it was evaluated, its limitations, and how it should be used in production.

In [ ]:
def format_metric(value):
    return f"{value:.4f}" if isinstance(value, (float, np.floating)) else str(value)


best_test_row = (
    test_results_df.loc[test_results_df["model"] == best_model_name]
    .iloc[0]
    .to_dict()
)

best_cv_row = (
    cv_results_df.loc[cv_results_df["model"] == best_model_name]
    .iloc[0]
    .to_dict()
)

model_card = {
    "model_card_version": "1.0",
    "model_name": "stage1_gatekeeper",
    "model_artifact": MODEL_OUTPUT_PATH,
    "model_type": "binary_text_classifier",
    "selected_model": best_model_name,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "owner": "lead_classifier_project",
    "intended_use": {
        "primary_use": "Classify inbound text messages as real-estate leads or spam/non-leads.",
        "positive_label": "1 = lead",
        "negative_label": "0 = spam_non_lead",
        "recommended_decision_policy": (
            "Use as a Stage 1 gatekeeper. Messages predicted as lead should proceed "
            "to downstream lead qualification or information extraction."
        ),
    },
    "out_of_scope_use": [
        "Do not use as a final sales qualification model.",
        "Do not use as a legal, financial, or housing eligibility decision system.",
        "Do not use on languages or markets not represented in the training data without validation.",
    ],
    "dataset": dataset_metadata,
    "features_and_preprocessing": {
        "input_columns": ["text"],
        "target_column": "label",
        "preprocessing": [
            "SimpleImputer fills missing text with an empty string.",
            "FunctionTransformer flattens text input for vectorization.",
            "TfidfVectorizer lowercases text, strips unicode accents, and uses unigrams + bigrams.",
        ],
        "tfidf_config": {
            "ngram_range": [1, 2],
            "max_features": 10000,
            "min_df": 2,
            "max_df": 0.95,
            "strip_accents": "unicode",
            "lowercase": True,
        },
    },
    "training": {
        "framework": "scikit-learn",
        "models_compared": list(models.keys()),
        "train_test_split": {
            "test_size": TEST_SIZE,
            "stratified": True,
            "random_state": RANDOM_STATE,
        },
        "cross_validation": {
            "method": "StratifiedKFold",
            "n_splits": CV_SPLITS,
            "shuffle": True,
            "random_state": RANDOM_STATE,
        },
        "model_selection_metric": "test_f1",
    },
    "evaluation": {
        "test_results_all_models": test_results_df.to_dict(orient="records"),
        "cv_results_all_models": cv_results_df.to_dict(orient="records"),
        "selected_model_test_metrics": best_test_row,
        "selected_model_cv_metrics": best_cv_row,
    },
    "limitations": [
        "The dataset is synthetic, so production language may differ from training examples.",
        "The model may over-rely on phrases seen in generated templates.",
        "False positives may send non-leads to downstream processing.",
        "False negatives may incorrectly block real leads.",
        "Performance should be revalidated once real app traffic is available.",
    ],
    "monitoring_recommendations": [
        "Log predictions, confidence where available, and human correction labels.",
        "Track lead false-negative rate using reviewed samples.",
        "Track spam/non-lead false-positive rate using reviewed samples.",
        "Review low-confidence or disputed predictions weekly.",
        "Retrain when real production examples materially differ from synthetic data.",
    ],
    "retraining_recommendations": [
        "Add real inbound messages after removing sensitive personal data.",
        "Add hard negatives that mention apartments but are not actionable leads.",
        "Add multilingual or transliterated examples if the app receives them.",
        "Re-run the same notebook and compare dataset hashes, F1, precision, and recall.",
    ],
    "runtime": {
        "python_version": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
    },
}


model_card_md = f"""# Model Card: Stage 1 Gatekeeper

## Model Details

- **Model name:** `stage1_gatekeeper`
- **Model type:** Binary text classifier
- **Selected model:** `{best_model_name}`
- **Artifact:** `{MODEL_OUTPUT_PATH}`
- **Created at UTC:** `{model_card["created_at_utc"]}`
- **Framework:** scikit-learn

## Intended Use

This model classifies inbound text messages for a real-estate lead gatekeeper.

- `0` = spam / non-lead
- `1` = real-estate lead

Recommended use: Stage 1 routing. Messages predicted as leads should continue to downstream lead qualification or information extraction.

## Out-of-Scope Use

- Not a final sales qualification model.
- Not a legal, financial, or housing eligibility decision system.
- Not validated for unseen languages, new markets, or channels without additional evaluation.

## Dataset Fingerprint

- **Dataset path:** `{dataset_metadata["dataset_path"]}`
- **Rows:** `{dataset_metadata["row_count"]}`
- **Columns:** `{dataset_metadata["column_names"]}`
- **Label distribution:** `{dataset_metadata["label_distribution"]}`
- **Raw file SHA-256:** `{dataset_metadata["dataset_raw_sha256"]}`
- **Loaded content SHA-256:** `{dataset_metadata["dataset_content_sha256"]}`
- **File size bytes:** `{dataset_metadata["dataset_file_size_bytes"]}`
- **Duplicate text/label rows:** `{dataset_metadata["duplicate_text_label_rows"]}`

## Features and Preprocessing

Input column: `text`

Pipeline:

1. `SimpleImputer(strategy="constant", fill_value="")`
2. `FunctionTransformer(np.ravel)`
3. `TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), max_features=10000, min_df=2, max_df=0.95)`
4. Classifier: `{best_model_name}`

## Training Setup

- **Test size:** `{TEST_SIZE}`
- **Split type:** Stratified train/test split
- **Random state:** `{RANDOM_STATE}`
- **Cross-validation:** StratifiedKFold, `{CV_SPLITS}` folds
- **Model selection metric:** test F1 score

## Evaluation Summary

### Selected Model Test Metrics

- **Accuracy:** {format_metric(best_test_row["test_accuracy"])}
- **Precision:** {format_metric(best_test_row["test_precision"])}
- **Recall:** {format_metric(best_test_row["test_recall"])}
- **F1:** {format_metric(best_test_row["test_f1"])}

### Selected Model Cross-Validation Metrics

- **CV Accuracy Mean:** {format_metric(best_cv_row["cv_accuracy_mean"])}
- **CV Precision Mean:** {format_metric(best_cv_row["cv_precision_mean"])}
- **CV Recall Mean:** {format_metric(best_cv_row["cv_recall_mean"])}
- **CV F1 Mean:** {format_metric(best_cv_row["cv_f1_mean"])}

## All Model Test Results

{test_results_df.to_markdown(index=False)}

## All Model Cross-Validation Results

{cv_results_df.to_markdown(index=False)}

## Limitations

- The dataset is synthetic, so production language may differ from training examples.
- The model may over-rely on phrases seen in generated templates.
- False positives may send non-leads to downstream processing.
- False negatives may incorrectly block real leads.
- Performance should be revalidated once real app traffic is available.

## Monitoring Recommendations

- Log predictions and reviewed outcomes.
- Track false negatives for real leads.
- Track false positives for spam/non-leads.
- Review low-confidence or disputed predictions.
- Retrain when real production examples diverge from synthetic data.

## Retraining Recommendations

- Add reviewed real inbound messages after removing sensitive personal data.
- Add hard negatives that mention apartments but are not actionable leads.
- Add multilingual, shorthand, and transliterated examples if users send them.
- Re-run the training notebook and compare dataset hashes and evaluation metrics.

## Runtime

- **Python:** `{sys.version.split()[0]}`
- **Platform:** `{platform.platform()}`
- **pandas:** `{pd.__version__}`
- **numpy:** `{np.__version__}`
- **scikit-learn:** `{sklearn.__version__}`
"""

Path(MODEL_CARD_MD_PATH).write_text(model_card_md, encoding="utf-8")
Path(MODEL_CARD_JSON_PATH).write_text(json.dumps(model_card, indent=2), encoding="utf-8")

# Update the saved model package with the model card paths and summary.
model_package["model_card"] = model_card
model_package["model_card_markdown_path"] = MODEL_CARD_MD_PATH
model_package["model_card_json_path"] = MODEL_CARD_JSON_PATH
joblib.dump(model_package, MODEL_OUTPUT_PATH)

print(f"Saved model card markdown to: {MODEL_CARD_MD_PATH}")
print(f"Saved model card JSON to: {MODEL_CARD_JSON_PATH}")
print(f"Updated model package with model card metadata: {MODEL_OUTPUT_PATH}")

## 14. Preview the model card

In [ ]:
print(Path(MODEL_CARD_MD_PATH).read_text(encoding="utf-8"))

## 13. Load the model and run production-style inference

In [ ]:
loaded_package = joblib.load(MODEL_OUTPUT_PATH)

loaded_model = loaded_package["model"]
loaded_label_mapping = loaded_package["label_mapping"]

new_messages = pd.DataFrame(
    {
        "text": [
            "Hi, is the apartment still available?",
            "URGENT claim your reward now",
            "Can you send pictures please?",
        ]
    }
)

predictions = loaded_model.predict(new_messages)

for message, prediction in zip(new_messages["text"], predictions):
    print("Message:", message)
    print("Prediction:", prediction, "-", loaded_label_mapping[prediction])
    print()


## Optional: download the trained model and model card from Colab

This downloads the model artifact, Markdown model card, and JSON model card.


In [ ]:
try:
    from google.colab import files
    files.download(MODEL_OUTPUT_PATH)
    files.download(MODEL_CARD_MD_PATH)
    files.download(MODEL_CARD_JSON_PATH)
except ImportError:
    print("Download helper only works inside Google Colab.")
